# Source Dashboard Overview

This notebook is executed by `dagstermill` from Dagster.
It uses `context.op_config` to choose source schema/table and chart columns.

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import dagstermill
from dagster import AssetMaterialization, MetadataValue

context = dagstermill.get_context()
cfg = context.op_config

unit_key = cfg["unit_key"]
db_schema = cfg["db_schema"]
db_table = cfg["db_table"]
chart_x_col = cfg["chart_x_col"]
chart_y_col = cfg["chart_y_col"]
row_limit = int(cfg.get("row_limit", 25))

run_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
repo_root = Path.cwd()
output_dir = repo_root / "analytics" / "dashboard_outputs" / unit_key.lower()
output_dir.mkdir(parents=True, exist_ok=True)

html_report_path = output_dir / f"{run_ts}_report.html"
chart_image_path = output_dir / f"{run_ts}_chart.png"

print(f"Dashboard unit: {unit_key}")
print(f"Reading {db_schema}.{db_table}")
print(f"Output dir: {output_dir}")

In [ ]:
conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
)

query = f"""
SELECT *
FROM {db_schema}.{db_table}
ORDER BY last_mtime DESC
LIMIT {row_limit}
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f"Fetched {len(df)} rows")
df.head()

In [ ]:
if df.empty:
    html_content = (
        f"<h2>{unit_key}</h2>"
        f"<p>No rows available in <code>{db_schema}.{db_table}</code>.</p>"
    )
    html_report_path.write_text(html_content, encoding="utf-8")
    print("No rows available to visualize.")
else:
    if chart_x_col not in df.columns or chart_y_col not in df.columns:
        raise ValueError(
            f"Configured chart columns not found. Available columns: {list(df.columns)}"
        )

    chart_df = df[[chart_x_col, chart_y_col]].copy()
    chart_df[chart_y_col] = pd.to_numeric(chart_df[chart_y_col], errors="coerce").fillna(0)

    plt.figure(figsize=(12, 6))
    plt.bar(chart_df[chart_x_col].astype(str), chart_df[chart_y_col])
    plt.title(f"{unit_key}: {chart_y_col} by {chart_x_col}")
    plt.xlabel(chart_x_col)
    plt.ylabel(chart_y_col)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(chart_image_path)
    plt.show()

    table_html = df.head(row_limit).to_html(index=False)
    html_content = f"""
    <html>
      <head><title>{unit_key} dashboard report</title></head>
      <body>
        <h2>{unit_key}</h2>
        <p>Source: <code>{db_schema}.{db_table}</code></p>
        <img src="{chart_image_path.name}" alt="dashboard chart" style="max-width: 100%;" />
        <h3>Latest rows</h3>
        {table_html}
      </body>
    </html>
    """
    html_report_path.write_text(html_content, encoding="utf-8")

dagstermill.yield_event(
    AssetMaterialization(
        asset_key=f"{unit_key.lower()}_dashboard_html_report",
        metadata={
            "report_path": MetadataValue.path(str(html_report_path)),
            "chart_path": MetadataValue.path(str(chart_image_path)),
            "row_count": len(df),
            "source_table": f"{db_schema}.{db_table}",
        },
    )
)

print(f"HTML report: {html_report_path}")
print(f"Chart image: {chart_image_path}")